# Financial and ESG portfolio optimisation

Configure the experiment in **Section 1**, then run collection, preparation, training, evaluation and reporting in order. Reusable calculations remain in `src/portfolio_experiment/`.

To regenerate a report from existing results without collecting data or training again, use **Section 7**. Data and completed runs remain in this project folder. Training always creates a new run, with separate chronological training, validation and testing periods.


## Setup

Run this notebook from the project folder using the Python environment installed from `requirements.txt`. The setup cell imports the experiment stages but does not collect data or start training.

In [2]:
from pathlib import Path
import sys
import json
from IPython.display import display, FileLink, Image

ROOT = Path.cwd().resolve()
if not (ROOT / "src" / "portfolio_experiment").is_dir():
    raise RuntimeError("Start Jupyter in the MAIN_NOTEBOOK_FOLDER project directory.")
sys.path.insert(0, str(ROOT / "src"))

from portfolio_experiment.config import ExperimentConfig
from portfolio_experiment import collect_data, prepare_study, range_table, train, evaluate, make_report
from portfolio_experiment.training import library_defaults

## 1. Experiment configuration

### 1.1 Profiles, model seeds and investor preferences

The profile selects a starting computational budget. Each resulting value remains directly editable beneath `BUDGETS`. Model seeds control independent PPO fits, while `FIRST_STOCK_SEED` in Section 1.2 controls stock selection. A larger **A** increases the variance penalty, and a larger **B** increases the preference for lower ESG risk. Use a tuple for a single value, for example `(0.01,)`.

In [ ]:
PROFILE = "smoke"  # "smoke", "pilot" or "mass"
BUDGETS = {
    "smoke": dict(model_seeds=(7,), training_steps=2_048,
                  baskets={"training": 1, "validation": 1, "test": 1},
                  checkpoint_frequency=2_048),
    "pilot": dict(model_seeds=(100, 101, 102), training_steps=50_000,
                  baskets={"training": 4, "validation": 1, "test": 2},
                  checkpoint_frequency=10_000),
    "mass": dict(model_seeds=tuple(range(100, 120)), training_steps=250_000,
                 baskets={"training": 20, "validation": 5, "test": 10},
                 checkpoint_frequency=50_000),
}

MODEL_SEEDS = BUDGETS[PROFILE]["model_seeds"]
TRAINING_STEPS = BUDGETS[PROFILE]["training_steps"]
WRDS_BASKETS = BUDGETS[PROFILE]["baskets"].copy()
CHECKPOINT_FREQUENCY = BUDGETS[PROFILE]["checkpoint_frequency"]
RISK_AVERSIONS = (3.0,) if PROFILE == "smoke" else (1.0, 3.0, 6.0)
ESG_APPETITES = (0.0, 0.005) if PROFILE == "smoke" else (0.0, 0.001, 0.005, 0.01, 0.1)

### 1.2 Data collection and evaluation periods

`DATA_SOURCE = "wrds"` collects data through WRDS. To reuse saved baskets, choose `"saved"` and supply their `data_manifest.json`. `REFRESH_WRDS_DATA` controls whether WRDS mode may use the local basket cache.

`PERIODS` defines separate chronological training, validation and testing dates. Earlier financial history initialises the features and covariance estimates. `RUN_TEST` controls whether the final evaluation includes the test period. Data manifests must remain inside this project.


In [ ]:
DATA_SOURCE = "wrds"  # "wrds" or "saved"
SAVED_DATA_MANIFEST = None  
REFRESH_WRDS_DATA = True
NUMBER_OF_ASSETS = 4
FIRST_STOCK_SEED = 250
CANDIDATE_POOL_SIZE = 400
MAXIMUM_COLLECTION_ATTEMPTS = 600  # Total attempts across all roles.
MINIMUM_PAST_OBSERVATIONS = 252
MAXIMUM_SCORE_AGE_DAYS = 120
PUBLICATION_LAG_BUSINESS_DAYS = 1

# WRDS table and field identifiers. The collector verifies the ESG field name.
FINANCIAL_TABLE = "crsp.dsf_v2"
SUSTAINABILITY_LIBRARY = "sustainalytics_all"
ESG_TABLE = "esgrr_focus"
ESG_FIELD_ID = 181110112399

FINANCIAL_START = "2018-01-02"
ESG_START = "2019-04-01"
END_DATE = "2025-12-31"
PERIODS = {
    "training": ("2019-04-01", "2022-12-30"),
    "validation": ("2023-01-03", "2023-12-29"),
    "test": ("2024-01-02", "2025-12-31"),
}
RUN_TEST = True

### 1.3 Financial features, portfolio rules and control optimisation

PPO and all controls use the same rebalance interval, transaction costs and starting value. Feature windows are counted in trading days. The correlation window also supplies the downside-volatility window, and relative momentum uses the longest momentum window. The solver tolerance and iteration limit control numerical convergence of the utility calculation.

In [ ]:
MOMENTUM_DAYS = (5, 20, 60, 252)
VOLATILITY_DAYS = (20, 60, 252)
CORRELATION_DAYS = 60
ESTIMATION_DAYS = 252

STARTING_VALUE = 10_000.0
TRANSACTION_COST = 0.0005 
REBALANCE_DAYS = 5
EPISODE_DAYS = 126
PERIODS_PER_YEAR = 252
ANNUAL_RISK_FREE_RATE = 0.0
REWARD_SCALE = 100.0

UTILITY_MAX_ITERATIONS = 20_000
UTILITY_TOLERANCE = 1e-9

### 1.4 PPO and checkpoint storage

`PPO_OVERRIDES = {}` uses the pinned Stable-Baselines3 defaults shown by the next cell. `HIDDEN_SIZE` sets the shared network width. Training uses validation utility to select a checkpoint and always saves initial, selected and final policies. `KEEP_CHECKPOINTS` additionally retains each candidate snapshot.


In [ ]:
PPO_OVERRIDES = {}
HIDDEN_SIZE = 64
DEVICE = "cpu"
CPU_THREADS = 2
KEEP_CHECKPOINTS = True
SHOW_FIGURES = True

display(library_defaults())

### 1.5 Build and inspect the configuration

Check the fit count, periods and complete configuration before collecting data. After changing an earlier setting, rerun these configuration cells before the experiment stages.


In [ ]:
cfg = ExperimentConfig(
    root=ROOT, profile=PROFILE,
    model_seeds=MODEL_SEEDS, training_steps=TRAINING_STEPS,
    baskets=WRDS_BASKETS, checkpoint_frequency=CHECKPOINT_FREQUENCY,
    risk_aversions=RISK_AVERSIONS, esg_appetites=ESG_APPETITES,
    data_source=DATA_SOURCE, saved_manifest=SAVED_DATA_MANIFEST,
    refresh_wrds=REFRESH_WRDS_DATA, number_of_assets=NUMBER_OF_ASSETS,
    first_stock_seed=FIRST_STOCK_SEED, candidate_pool_size=CANDIDATE_POOL_SIZE,
    maximum_collection_attempts=MAXIMUM_COLLECTION_ATTEMPTS,
    financial_table=FINANCIAL_TABLE, sustainability_library=SUSTAINABILITY_LIBRARY,
    esg_table=ESG_TABLE, esg_field_id=ESG_FIELD_ID,
    financial_start=FINANCIAL_START, esg_start=ESG_START, end_date=END_DATE,
    minimum_past_observations=MINIMUM_PAST_OBSERVATIONS,
    maximum_score_age_days=MAXIMUM_SCORE_AGE_DAYS,
    publication_lag_business_days=PUBLICATION_LAG_BUSINESS_DAYS,
    periods=PERIODS, run_test=RUN_TEST,
    rebalance_days=REBALANCE_DAYS, episode_days=EPISODE_DAYS,
    starting_value=STARTING_VALUE, transaction_cost=TRANSACTION_COST,
    periods_per_year=PERIODS_PER_YEAR, annual_risk_free_rate=ANNUAL_RISK_FREE_RATE,
    estimation_days=ESTIMATION_DAYS, momentum_days=MOMENTUM_DAYS,
    volatility_days=VOLATILITY_DAYS, correlation_days=CORRELATION_DAYS,
    utility_max_iterations=UTILITY_MAX_ITERATIONS, utility_tolerance=UTILITY_TOLERANCE,
    reward_scale=REWARD_SCALE, ppo_overrides=PPO_OVERRIDES,
    hidden_size=HIDDEN_SIZE, device=DEVICE, cpu_threads=CPU_THREADS,
    keep_checkpoints=KEEP_CHECKPOINTS, show_figures=SHOW_FIGURES,
).validate()

print(f"Profile: {cfg.profile} | {len(cfg.preferences) * len(cfg.model_seeds)} independent fits")
print(f"Requested training transitions per fit: {cfg.training_steps:,}")
print(f"Data: {cfg.data_source} | Rebalance: {cfg.rebalance_days} trading days")
print(json.dumps(cfg.as_dict(), indent=2))

## 2. Collect financial and ESG records

`wrds_data.py` retrieves daily returns and overall ESG Risk Scores, links financial and rating identifiers, and applies the availability rules. It records accepted and rejected basket seeds and query/cache status under `data/collections/`. Keep the printed collection path for later reuse.

In [ ]:
files = collect_data(cfg)

## 3. Prepare the baskets and observations

`data.py` calculates historical features and sample covariance, enforces stock and issuer separation, and fits standardisation on training observations. Validation and testing use that same fitted transformation. The table lists the accepted baskets and evaluated return dates.

In [ ]:
study = prepare_study(cfg, files)
display(range_table(study, cfg))

## 4. Train PPO and select checkpoints

`policy.py` defines the shared stock-processing network, `environment.py` applies allocations and daily accounting, and `finance.py` supplies the reward. `training.py` fits one model per A/B preference and seed, periodically evaluates it on validation baskets, and saves initial, selected and final policies.

Validation selects a saved trained policy without updating its parameters. Training continues to its specified length, and testing results do not enter checkpoint selection.

In [ ]:
run_folder = train(cfg, study)
print("Saved run:", run_folder)

## 5. Evaluate PPO and the controls

`evaluation.py` runs the saved policies and controls over each complete period using common accounting. Controls comprise static and rolling ESG utility, static and rolling financial utility, and equal weighting. Financial-control allocations use B = 0, while their reported utility is evaluated under the preference being compared. The PPO at B = 0 still receives ESG observations but has no ESG reward term.

`RUN_TEST = False` omits test-role evaluation. It does not omit the training or validation evaluations.

In [ ]:
results = evaluate(cfg, study, run_folder)
import pandas as pd
display(pd.read_csv(run_folder / "summary.csv"))

## 6. Generate the results report

The PDF contains parameters, outcome tables and the results figures used in the thesis, including the larger combined heatmaps, training-versus-testing utility and overall portfolio paths. Utility is shown in its original units. Comparable heatmaps share one colour scale per metric, and return differences use percentage points.

Each figure is exported as PNG, SVG and PDF. Plotted values and the thesis figure mapping are recorded alongside the figures. Section 7 regenerates the same outputs from a completed run.


In [ ]:
report = make_report(run_folder)
display(FileLink(str(report["pdf"])))
if cfg.show_figures:
    for path in report["figures"]:
        display(Image(filename=str(path)))

## 7. Regenerate a report from saved results

Run the setup cell, set `SAVED_REPORT_FOLDER` below, then run only this cell. It rebuilds figures and the results PDF from completed CSV files without WRDS collection, training or policy evaluation. Leave it as `None` during a normal Run All.


In [5]:
SAVED_REPORT_FOLDER = ROOT/"reports"/"20261019_081443_316823Z"  # For example: ROOT / "reports" / "RUN_ID"
if SAVED_REPORT_FOLDER is not None:
    saved_report = make_report(Path(SAVED_REPORT_FOLDER))
    display(FileLink(str(saved_report["pdf"])))

PDF report: /Users/samuelville/Desktop/University_Of_Sydney/Subjects/2026S2/Thesis_2/code/MAIN_NOTEBOOK_FOLDER/reports/20261019_081443_316823Z/MAIN_performance_report.pdf


/Users/samuelville/Desktop/University_Of_Sydney/Subjects/2026S2/Thesis_2/code/MAIN_NOTEBOOK_FOLDER/reports/20261019_081443_316823Z/MAIN_performance_report.pdf